# 🧠 Facial Emotion Recognition — Kaggle Training Notebook

Trains a CNN on 48×48 grayscale face images across **7 emotion classes**.

**Output:** `facialemotionmodel.h5` saved to `/kaggle/working/`

### ⚙️ Before running:
1. Click the **three dots (⋮)** at top right → **Accelerator** → select **GPU T4 x2** or **P100**
2. Upload your dataset: go to the **Data** tab on the right → **Upload** your `images.zip` (containing `train/` and `test/` folders)
3. Once uploaded, your dataset will be available at `/kaggle/input/<your-dataset-name>/`
4. Update `DATASET_NAME` in Step 1 below to match your dataset name (shown in the Data tab)

## Step 1 — Configuration

In [ ]:
import os

# ── UPDATE THIS to match your dataset name in the Kaggle Data tab ────────────
# e.g. if your dataset shows as '/kaggle/input/facial-emotion-images/' use:
# DATASET_NAME = 'facial-emotion-images'
DATASET_NAME = 'facial-emotion-images'  # <-- change this

# Paths
INPUT_DIR  = f'/kaggle/input/{DATASET_NAME}'
OUTPUT_DIR = '/kaggle/working'
TRAIN_DIR  = os.path.join(INPUT_DIR, 'images', 'train')
TEST_DIR   = os.path.join(INPUT_DIR, 'images', 'test')

# Fallback: if images/ subfolder doesn't exist, try root of dataset
if not os.path.exists(TRAIN_DIR):
    TRAIN_DIR = os.path.join(INPUT_DIR, 'train')
    TEST_DIR  = os.path.join(INPUT_DIR, 'test')

print('Train dir:', TRAIN_DIR, '— exists:', os.path.exists(TRAIN_DIR))
print('Test  dir:', TEST_DIR,  '— exists:', os.path.exists(TEST_DIR))
print('Output dir:', OUTPUT_DIR)

# Show what Kaggle can see in the input folder
print('\nAvailable datasets in /kaggle/input/:')
for d in os.listdir('/kaggle/input'):
    print(' ', d)

## Step 2 — Check GPU

In [ ]:
import tensorflow as tf

print('TensorFlow version:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPUs available:', gpus)

if gpus:
    print('✅ GPU detected — training will be fast!')
    # Allow memory growth to avoid OOM errors
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
else:
    print('⚠️  No GPU detected. Go to Settings → Accelerator → GPU T4 x2')

## Step 3 — Imports

In [ ]:
import glob
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from tensorflow.keras.preprocessing.image import load_img
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import (
    Dense, Conv2D, Dropout, Flatten,
    MaxPooling2D, BatchNormalization
)
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
print('All imports successful.')

## Step 4 — Build Dataset Dataframes

In [ ]:
def createdataframe(dir):
    image_paths, labels = [], []
    for label in os.listdir(dir):
        label_dir = os.path.join(dir, label)
        if not os.path.isdir(label_dir):
            continue
        for imagename in os.listdir(label_dir):
            image_paths.append(os.path.join(label_dir, imagename))
            labels.append(label)
        print(label, 'completed —', len(os.listdir(label_dir)), 'images')
    return image_paths, labels

train_df = pd.DataFrame()
train_df['image'], train_df['label'] = createdataframe(TRAIN_DIR)

test_df = pd.DataFrame()
test_df['image'], test_df['label'] = createdataframe(TEST_DIR)

print('\nTrain samples:', len(train_df))
print('Test  samples:', len(test_df))
print('Classes:', sorted(train_df['label'].unique()))

## Step 5 — Extract Features & Normalise

In [ ]:
def extract_features(images):
    features = []
    for image in tqdm(images):
        img = load_img(image, color_mode='grayscale', target_size=(48, 48))
        features.append(np.array(img))
    return np.array(features).reshape(-1, 48, 48, 1)

print('Extracting train features...')
x_train = extract_features(train_df['image']) / 255.0

print('Extracting test features...')
x_test = extract_features(test_df['image']) / 255.0

print('x_train shape:', x_train.shape)
print('x_test  shape:', x_test.shape)

## Step 6 — Encode Labels

In [ ]:
le = LabelEncoder()
le.fit(train_df['label'])

y_train = to_categorical(le.transform(train_df['label']), num_classes=7)
y_test  = to_categorical(le.transform(test_df['label']),  num_classes=7)

print('Label classes:', list(le.classes_))
print('y_train shape:', y_train.shape)

## Step 7 — Build or Resume Model

> **Note:** Kaggle sessions reset between runs, so checkpoints in `/kaggle/working/checkpoints/` are preserved as long as you don't clear outputs. If starting fresh, the model builds from scratch.

In [ ]:
checkpoint_dir = os.path.join(OUTPUT_DIR, 'checkpoints')
os.makedirs(checkpoint_dir, exist_ok=True)

checkpoint_files = glob.glob(os.path.join(checkpoint_dir, '*.h5'))
start_epoch = 0

if checkpoint_files:
    latest_checkpoint = max(checkpoint_files, key=os.path.getmtime)
    print('Resuming from:', latest_checkpoint)
    model = load_model(latest_checkpoint, compile=False)
    model.compile(
        optimizer=Adam(learning_rate=1e-4),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    match = re.search(r'facialemotionmodel(\d+)_', os.path.basename(latest_checkpoint))
    if match:
        start_epoch = int(match.group(1))
    print('Resuming from epoch:', start_epoch)

else:
    print('No checkpoint found — building model from scratch')
    model = Sequential([
        tf.keras.layers.Input(shape=(48, 48, 1)),

        Conv2D(64,  (3, 3), padding='same', activation='relu'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),   # → 24×24
        Dropout(0.25),

        Conv2D(128, (3, 3), padding='same', activation='relu'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),   # → 12×12
        Dropout(0.25),

        Conv2D(256, (3, 3), padding='same', activation='relu'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),   # → 6×6
        Dropout(0.25),

        Flatten(),
        Dense(512, activation='relu'),
        BatchNormalization(),
        Dropout(0.4),
        Dense(256, activation='relu'),
        Dropout(0.3),
        Dense(7, activation='softmax')
    ])
    model.compile(
        optimizer=Adam(learning_rate=1e-3),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

model.summary()

## Step 8 — Train

In [ ]:
checkpoint_cb = ModelCheckpoint(
    filepath=os.path.join(checkpoint_dir, 'facialemotionmodel{epoch:02d}_valacc{val_accuracy:.4f}.h5'),
    monitor='val_accuracy',
    save_best_only=True,
    save_weights_only=False,
    verbose=1
)

reduce_lr_cb = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=1
)

history = model.fit(
    x=x_train, y=y_train,
    batch_size=64,
    epochs=100,
    initial_epoch=start_epoch,
    validation_data=(x_test, y_test),
    callbacks=[checkpoint_cb, reduce_lr_cb]
)

## Step 9 — Save Final Model

In [ ]:
# Save all model files to /kaggle/working/ so you can download them
model_json = model.to_json()
with open(os.path.join(OUTPUT_DIR, 'facialemotionmodel.json'), 'w') as f:
    f.write(model_json)

model.save_weights(os.path.join(OUTPUT_DIR, 'facialemotionmodel.weights.h5'))
model.save(os.path.join(OUTPUT_DIR, 'facialemotionmodel.h5'))

print('✅ Files saved to /kaggle/working/:')
for f in os.listdir(OUTPUT_DIR):
    fpath = os.path.join(OUTPUT_DIR, f)
    if os.path.isfile(fpath):
        size_mb = os.path.getsize(fpath) / (1024 * 1024)
        print(f'  {f}  ({size_mb:.1f} MB)')

## Step 10 — Plot Training History

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'],     label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'],     label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'training_history.png'))
plt.show()
print('Training complete!')